# SpaNCy-Shift-DL Explorer

Single-stage deep learning batch normalization for CyCIF multiplexed imaging.

**Single DL stage:** `ResidualShiftModel` trained directly on raw log1p data with:
- `L_ref` (w=1.0, epoch 0): per-sample alignment to KL-medoid reference anchors → 1D histogram alignment
- `L_mmd` (w=0.5, ramped): 20D RBF MMD across batches → multivariate mixing for kBET
- `L_recon` (w=0.1): Huber to identity → keeps shifts small, biology-preserving

Unlike `spancy_shift.py` (two-stage), there is **no analytic Stage 1**. Reference alignment is
learned via `L_ref`. The KL-medoid reference selection and bimodal detection remain analytic
(cheap, one-time data statistics — not normalization).

**Sections:**
0. Colab Setup
1. Load & Inspect Data
2. Bimodal Marker Detection
2b. Reference Sample Selection
3. Training (single-stage DL)
4. Inference & Histogram Inspection
5. Batch adj-R² Diagnostics
6. Positive Population Preservation Check
7. Histogram Comparison
8. kBET Evaluation

## 0. Colab Setup

Run first on Google Colab. Installs dependencies and uploads `spancy_shift_dl.py`.
No torch-geometric needed — this module uses plain PyTorch only.

In [ ]:
!pip install -q anndata scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift_dl.py
from google.colab import files
uploaded = files.upload()

import os
assert os.path.exists('spancy_shift_dl.py'), 'spancy_shift_dl.py not found — upload it first'
print(f"spancy_shift_dl.py uploaded ({os.path.getsize('spancy_shift_dl.py'):,} bytes)")

In [ ]:
import importlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt

import spancy_shift_dl
from spancy_shift_dl import (
    load_adata, log1p_scale,
    detect_bimodal_markers, find_best_sample_per_marker,
    compute_reference_anchors,
    ResidualShiftModel, train,
    normalize_adata,
    positive_population_table, per_marker_batch_r2,
)

# After re-uploading spancy_shift_dl.py:
# importlib.reload(spancy_shift_dl); from spancy_shift_dl import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Download PRAD dataset
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")
print(f"\nobs columns: {list(adata.obs.columns)}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)

In [ ]:
# Per-batch mean intensity overview
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].boxplot([X_raw[:, i] for i in range(X_raw.shape[1])], labels=marker_names, vert=True)
axes[0].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[0].set_title('Raw marker distributions (all cells)')
axes[0].set_ylabel('Intensity')

for b in unique_batches:
    mask = batch_vals == b
    axes[1].plot(X_raw[mask].mean(axis=0), marker='o', markersize=3, label=str(b), alpha=0.8)
axes[1].set_xticks(range(len(marker_names)))
axes[1].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[1].set_title('Per-batch mean intensity')
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Bimodal Marker Detection (preview on raw data)

Adaptive per-batch voting: a marker is bimodal if ≥50% of batches independently show ≥2
prominent histogram peaks.

- **Bimodal**: separate neg/pos shifts blended by sigmoid in the DL model; separate neg/pos
  anchors in L_ref; bimodal dims masked from MMD
- **Unimodal**: single mean shift; single anchor in L_ref

Detection runs on `log1p(X_raw)`. Inside `train()`, thresholds are automatically converted
to scaled space via `scaler.center_` / `scaler.scale_` for use in `ResidualShiftModel`.

In [ ]:
# Preview: log1p raw data for bimodal detection visualization
X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
print(f'log1p range: [{X_log_preview.min():.3f}, {X_log_preview.max():.3f}]')

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"\nBimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")

print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")

## 2b. Reference Sample Selection

For each marker, find the **medoid sample** — the one with lowest mean pairwise symmetric KL
divergence to all other samples in log1p space. This sample's distribution becomes the
normalization target for that marker via `L_ref`.

The medoid is a data-driven reference (not a global mean/quantile). Different markers can
have different reference samples.

In [ ]:
# Compute reference sample per marker (passed to train() to skip recomputation)
ref_sample_per_marker = find_best_sample_per_marker(adata)

# Show as table
ref_df = pd.DataFrame(
    [(m, s) for m, s in ref_sample_per_marker.items()],
    columns=['marker', 'reference_sample'],
)
print('Per-marker reference sample (normalization target):\n')
print(ref_df.to_string(index=False))

ref_counts = ref_df['reference_sample'].value_counts()
print(f'\nDistinct reference samples used: {len(ref_counts)}')
print(ref_counts.to_string())

# Bar chart: how many markers use each sample as reference
fig, ax = plt.subplots(figsize=(14, 3))
sample_order = sorted(adata.obs['sample_id'].unique())
ref_matrix = np.zeros(len(sample_order))
for i, s in enumerate(sample_order):
    ref_matrix[i] = sum(v == s for v in ref_sample_per_marker.values())
ax.bar(range(len(sample_order)), ref_matrix, color='steelblue', alpha=0.85)
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, rotation=45, ha='right')
ax.set_ylabel('# markers using as reference')
ax.set_title('How often each sample is chosen as reference (KL medoid across markers)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize histograms with detected bimodal thresholds
from scipy.ndimage import gaussian_filter1d

n_cols = 5
n_rows = int(np.ceil(len(marker_names) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

for m, mname in enumerate(marker_names):
    ax = axes[m]
    col = X_log_preview[:, m]
    lo, hi = np.percentile(col, [1, 99])
    bins = np.linspace(lo, hi, 151)
    counts, edges = np.histogram(col, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    smoothed = gaussian_filter1d(counts.astype(float), sigma=2)

    ax.fill_between(centers, smoothed, alpha=0.4,
                    color='steelblue' if marker_is_bimodal[m] else 'salmon')
    ax.plot(centers, smoothed, 'k-', linewidth=1)

    if marker_is_bimodal[m]:
        ax.axvline(thresholds[m], color='red', linestyle='--', linewidth=1.5,
                   label=f'thresh={thresholds[m]:.2f}')
        ax.legend(fontsize=6)

    label = 'BIMODAL' if marker_is_bimodal[m] else 'unimodal'
    ax.set_title(f'{mname}\n({label})', fontsize=8,
                 fontweight='bold' if marker_is_bimodal[m] else 'normal')
    ax.set_xlabel('log1p value', fontsize=7)
    ax.tick_params(labelsize=6)

for ax in axes[len(marker_names):]:
    ax.set_visible(False)

plt.suptitle('Marker Distributions — Bimodal Detection (log1p space, raw data)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Training (single-stage DL)

Trains `ResidualShiftModel` directly on raw log1p data. No analytic Stage 1.

**Inside `train()`:**
1. Fits `RobustScaler` on `log1p(X_raw)`
2. Detects bimodal markers on `log1p(X_raw)`; converts thresholds to scaled space
3. Precomputes reference anchors (soft-weighted means of reference sample in scaled space)
4. Trains `ResidualShiftModel` with:
   - `L_ref` (w=1.0, active from epoch 0) — pull each sample's neg/pos means toward reference anchors
   - `L_mmd` (w=0.5, ramped from `mmd_ramp_start`) — 20D MMD for multivariate batch mixing
   - `L_recon` (w=0.1) — Huber to identity (keeps shifts small)

**Key parameters:**
- `w_ref=1.0` — main 1D alignment driver (replaces analytic Stage 1)
- `mmd_mask_pospop=True` — clamp bimodal dims to threshold before MMD (prevents biology → batch confusion)
- `sharpness=10.0` — sigmoid steepness for bimodal neg/pos blending
- `max_shift=0.5` — caps shift magnitude in RobustScaler units (≈0.5 IQR)

In [ ]:
N_EPOCHS = 10  # quick test; use 50-100 for production

model, trained_scaler, trained_ref, history = train(
    adata,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    cells_per_step=16000,
    device_str=DEVICE,
    warmup_epochs=2,
    mmd_ramp_start=2,
    mmd_ramp_end=8,
    w_ref=1.0,            # L_ref weight — main 1D alignment driver
    w_mmd=0.5,            # L_mmd weight (reduced from spancy_shift.py's 1.0)
    w_recon=0.1,          # L_recon weight (reduced — max_shift already constrains magnitude)
    mmd_samples=256,
    mmd_mask_pospop=True,
    sharpness=10.0,
    max_shift=0.5,
    bimodal_min_batch_frac=0.5,
    ref_sample_per_marker=ref_sample_per_marker,
)

print(f'\nTraining complete.')
print(f'  Final loss={history["loss"][-1]:.4f}  '
      f'ref={history["ref"][-1]:.4f}  '
      f'mmd={history["mmd"][-1]:.4f}  recon={history["recon"][-1]:.4f}')

In [ ]:
# Loss curves — 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: Total loss
axes[0].plot(history['loss'], 'k-', linewidth=1.5)
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)

# Panel 2: Component losses + MMD ramp
axes[1].plot(history['ref'],   'b-',   linewidth=1.5, label='L_ref (×w_ref)')
axes[1].plot(history['mmd'],   'r--',  linewidth=1.5, label='L_mmd (×mmd_w)')
axes[1].plot(history['recon'], color='darkorange', linewidth=1.5, linestyle=':', label='L_recon (×w_recon)')
ax_w = axes[1].twinx()
ax_w.plot(history['mmd_weight'], 'g--', linewidth=1, alpha=0.5, label='MMD weight')
ax_w.set_ylabel('MMD weight', color='green', fontsize=9)
ax_w.tick_params(axis='y', labelcolor='green')
axes[1].set_title('Component Losses + MMD Ramp')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, alpha=0.3)

# Panel 3: Learning rate
axes[2].plot(history['lr'], color='purple', linewidth=1.5)
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.suptitle('DL Training — ResidualShiftModel (single-stage)', fontsize=13)
plt.tight_layout()
plt.show()

# Shift magnitudes
with torch.no_grad():
    s_neg = model.shift_neg.weight.cpu().numpy()
    s_pos = model.shift_pos.weight.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, data, title in zip(axes, [s_neg, s_pos], ['|shift_neg|', '|shift_pos|']):
    im = ax.imshow(np.abs(data), aspect='auto', cmap='YlOrRd')
    ax.set_xlabel('Marker index')
    ax.set_ylabel('Sample index')
    ax.set_xticks(range(len(marker_names)))
    ax.set_xticklabels(marker_names, rotation=90, fontsize=7)
    ax.set_title(f'{title} per sample × marker')
    plt.colorbar(im, ax=ax, label='|shift|')

plt.suptitle('Learned Shifts (in RobustScaler space)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Inference & Output Inspection

In [ ]:
# Single-stage inference: raw → scale → model → inverse scale → expm1
adata_norm = normalize_adata(
    adata, model, trained_scaler, trained_ref,
    device_str=DEVICE,
    layer_name='normalized',
)

X_norm = np.asarray(adata_norm.layers['normalized'])
print(f'Normalized: min={X_norm.min():.4f}  max={X_norm.max():.4f}  mean={X_norm.mean():.4f}')
print(f'Layers: {list(adata_norm.layers.keys())}')

In [ ]:
# Compare raw vs normalized for bimodal markers
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]
plot_markers = (bimodal_names[:4] if len(bimodal_names) >= 4
                else bimodal_names + marker_names[:max(0, 4 - len(bimodal_names))])

layers_to_show = [
    (None,          'Raw'),
    ('normalized',  'DL Normalized'),
]

fig, axes = plt.subplots(len(plot_markers), len(layers_to_show),
                          figsize=(7 * len(layers_to_show), 4 * len(plot_markers)))
if len(plot_markers) == 1:
    axes = axes.reshape(1, -1)

for row, mname in enumerate(plot_markers):
    m_idx = marker_names.index(mname)
    for col, (layer_key, title) in enumerate(layers_to_show):
        ax = axes[row, col]
        if layer_key is None:
            data = X_raw
        else:
            data = adata_norm.layers[layer_key]
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(np.asarray(data)[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {title}')
        ax.set_xlabel('log1p intensity')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, title='Batch', ncol=2)

plt.suptitle('Bimodal Marker Distributions: Raw → DL Normalized', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Batch adj-R² Diagnostics

Adjusted R² for each marker regressed on batch labels.
Lower = batch effect better removed.

In [ ]:
batch_arr = adata_norm.obs['batch_id'].values
X_norm_arr = np.asarray(adata_norm.layers['normalized'])

r2_raw  = per_marker_batch_r2(np.log1p(np.clip(X_raw, 0, None)),      batch_arr, marker_names)
r2_norm = per_marker_batch_r2(np.log1p(np.clip(X_norm_arr, 0, None)), batch_arr, marker_names)

r2_compare = (
    r2_raw .rename(columns={'adj_r2': 'raw'})
    .merge(r2_norm.rename(columns={'adj_r2': 'normalized'}), on='marker')
)

print('Per-Marker Batch adj-R² (lower = better):\n')
print(r2_compare.to_string(index=False, float_format='%.4f'))
print(f'\nMean adj-R²:')
for col in ['raw', 'normalized']:
    print(f'  {col:>12s}: {r2_compare[col].mean():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(r2_compare))
w = 0.35
ax.bar(x - w/2, r2_compare['raw'],        w, label='Raw',        color='salmon',      alpha=0.85)
ax.bar(x + w/2, r2_compare['normalized'], w, label='Normalized', color='forestgreen', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(r2_compare['marker'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Adjusted R²')
ax.set_title('Per-Marker Batch adj-R² — lower is better')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='Target (0.05)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Positive Population Preservation Check

% positive cells per marker per sample using Otsu's threshold.
Delta = normalized − raw. Target: |delta| < 5% per marker.

In [ ]:
pop_table = positive_population_table(
    adata_norm, norm_layer='normalized', sample_col='sample_id'
)

summary = pop_table.groupby('marker').agg(
    mean_pct_raw=('pct_pos_raw', 'mean'),
    mean_pct_norm=('pct_pos_norm', 'mean'),
    mean_abs_delta=('delta', lambda x: x.abs().mean()),
    max_abs_delta=('delta', lambda x: x.abs().max()),
).round(2)

print('Positive Population Preservation (DL normalized vs raw):')
print('=' * 65)
print(summary.to_string())
print(f'\nOverall mean |delta|: {pop_table["delta"].abs().mean():.2f}%')
print(f'Overall max  |delta|: {pop_table["delta"].abs().max():.2f}%')
print(f'\nTarget: |delta| < 5% per marker  (UniFORM: -3.4% mean but large per-marker distortions)')

In [ ]:
pivot = pop_table.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: DL Normalized vs Raw')
plt.tight_layout()
plt.show()

In [ ]:
plot_markers_pop = bimodal_names[:4] if len(bimodal_names) >= 4 else bimodal_names[:2]

fig, axes = plt.subplots(1, max(len(plot_markers_pop), 1),
                          figsize=(5 * max(len(plot_markers_pop), 1), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table[pop_table['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x (no change)')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (DL Normalized)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation (DL Normalized vs Raw)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Histogram Comparison

Per-sample histogram PDFs for all markers. Saved to `histograms_shift_dl/`.

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps

save_dir = 'histograms_shift_dl'
os.makedirs(save_dir, exist_ok=True)

layers_to_plot = [
    ('normalized', 'DL Normalized'),
]

sample_col = 'sample_id' if 'sample_id' in adata_norm.obs.columns else 'batch_id'
sample_ids = adata_norm.obs[sample_col].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
cmap = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

pdf_path = os.path.join(save_dir, 'shift_dl_histograms.pdf')
rows_per_page = 4
n_cols = 2  # raw + normalized side by side

with PdfPages(pdf_path) as pdf:
    page_markers = []
    for i, mname in enumerate(marker_names):
        page_markers.append(mname)
        if len(page_markers) == rows_per_page or i == len(marker_names) - 1:
            fig, axes = plt.subplots(len(page_markers), n_cols,
                                     figsize=(7 * n_cols, 4 * len(page_markers)))
            if len(page_markers) == 1:
                axes = axes.reshape(1, -1)

            panels = [(None, 'Raw')] + layers_to_plot
            for row, mname_p in enumerate(page_markers):
                m_idx = marker_names.index(mname_p)
                for col, (layer_key, label) in enumerate(panels):
                    ax = axes[row, col]
                    if layer_key is None:
                        X_col = X_raw[:, m_idx]
                    else:
                        X_col = np.asarray(adata_norm.layers[layer_key][:, m_idx]).ravel()
                    X_log = np.log1p(np.clip(X_col, 0, None))
                    for s in unique_samples:
                        mask = sample_ids == s
                        c, e = np.histogram(X_log[mask], bins=80)
                        ax.plot(e[:-1], c, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
                    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
                    ax.set_title(f'{mname_p}{bim_tag} — {label}', fontsize=9)
                    ax.set_xlabel('log1p intensity', fontsize=8)
                    if col == 0:
                        ax.set_ylabel('Count', fontsize=8)
                    if row == 0 and col == 0:
                        ax.legend(fontsize=5, ncol=3, frameon=False)
                    ax.spines[['top', 'right']].set_visible(False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            page_markers = []

print(f'PDF saved to: {pdf_path}')

## 8. kBET Evaluation

Compute kBET acceptance rate for 5 clinical groups pairing samples from different batches.

- **Higher kBET = better** (batches well-mixed in local neighborhoods)
- **Lower chi-square = better**

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

EVAL_LAYERS = ['normalized']
print(f'Evaluating: {EVAL_LAYERS}')
print()
print('Baselines (g1–g5 mean kBET):')
print('  shift_normalize.py (pure scipy): 0.6307')
print('  spancy_shift.py Stage 1 (equiv): ~0.631')
print('  UniFORM:                         0.631')
print('  SpaNCy-GNN ensemble hybrid:      0.574')
print()
print('Target: kBET > 0.631 (beat UniFORM) with |Δ positive pop| < 5% per marker')

In [ ]:
all_kbet = {}

for layer_name in EVAL_LAYERS:
    print(f'\n{"-"*55}\n  {layer_name}\n{"-"*55}')
    adata_kbet = adata_norm.copy()
    adata_kbet.X = adata_norm.layers[layer_name].copy()

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}

    all_kbet[layer_name] = layer_res

print('\nDone.')

In [ ]:
print('=' * 70)
print('kBET Summary (mean across groups — higher = better)')
print('=' * 70)

summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    if kbets:
        row = {
            'layer': layer_name,
            'mean_kBET': float(np.mean(kbets)),
            'mean_chi2': float(np.mean(chi2s)),
            'n_groups': f'{len(kbets)}/{len(res)}',
        }
        summary_rows.append(row)
        print(f"  {layer_name:>20s}  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  ({row['n_groups']} groups)")

print()
print('Baselines:')
print('  shift_normalize.py / Stage 1: kBET~0.631')
print('  UniFORM:                      kBET=0.631')
print('  DL target:                    kBET>0.631  (L_ref + MMD improvement)')

df_kbet = pd.DataFrame(summary_rows)

if len(df_kbet) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].bar(df_kbet['layer'], df_kbet['mean_kBET'], color='forestgreen', alpha=0.85)
    axes[0].axhline(0.631, color='red', linestyle='--', linewidth=1.5, label='UniFORM 0.631')
    axes[0].set_ylabel('Mean kBET acceptance rate')
    axes[0].set_title('kBET (higher = better)')
    axes[0].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_ylim(0, 1)

    axes[1].bar(df_kbet['layer'], df_kbet['mean_chi2'], color='forestgreen', alpha=0.85)
    axes[1].set_ylabel('Mean chi-square')
    axes[1].set_title('Chi-square (lower = better)')
    axes[1].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()